# **2. Формирование признакового пространства (Feature Engineering): построение причинно-корректных признаков транспортной нагрузки**

* __Цель формирования признакового пространства:__ преобразовать подготовленный почасовой набор данных в таблицу признаков для последующего хронологического разбиения и обучения регрессионных моделей.
* __Задачи формирования признакового пространства:__
  - сформировать временные и календарные признаки;
  - выполнить циклическое кодирование периодических компонентов времени;
  - создать lag- и rolling-признаки только из прошлых значений `traffic_volume`;
  - удалить технически неполные строки.
* __Алгоритм выполнения:__
  1. __Загрузка данных:__ получение регулярного почасового набора через `load_raw_data()`.
  2. __Формирование признаков:__ последовательный вызов `build_feature_dataset()`.
  3. __Группировка признаков:__ выделение временных, календарных, погодных, lag- и rolling-компонентов.
  4. __Проверка причинности:__ сопоставление historical features с наблюдениями, предшествующими прогнозируемому интервалу.
  5. __Контроль результата:__ проверка размерности, хронологии и отсутствия технических пропусков в target и historical features.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import (
    DATETIME_COLUMN,
    LAG_HOURS,
    REPORTS_DIR,
    ROLLING_WINDOWS,
    TARGET_COLUMN,
)
from traffic_forecasting.data_loader import load_raw_data
from traffic_forecasting.features import (
    HISTORICAL_FEATURE_COLUMNS,
    LAG_FEATURE_COLUMNS,
    build_feature_dataset,
    get_feature_groups,
)

TABLES_DIR = REPORTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## **2.1. Загрузка подготовленных данных (Load Prepared Data)**

In [ ]:
# Use the ingestion layer so hourly gaps remain explicit before feature creation.
raw_data = load_raw_data()

display(Markdown("### **Размерность подготовленного набора данных (Prepared Dataset Shape)**"))
display(pd.DataFrame({"rows": [raw_data.shape[0]], "columns": [raw_data.shape[1]]}))
display(raw_data.head())

## **2.2. Построение полного набора признаков (Build Complete Feature Dataset)**

In [ ]:
feature_data = build_feature_dataset(raw_data)

shape_comparison = pd.DataFrame(
    {
        "stage": ["prepared_raw_data", "feature_dataset"],
        "rows": [raw_data.shape[0], feature_data.shape[0]],
        "columns": [raw_data.shape[1], feature_data.shape[1]],
    }
)

shape_comparison.to_csv(TABLES_DIR / "feature_engineering_shape_comparison.csv", index=False)

display(Markdown("### **Размерность до и после формирования признаков (Before/After Shape)**"))
display(shape_comparison)
display(Markdown("### **Первые строки набора признаков (Feature Dataset Preview)**"))
display(feature_data.head())

## **2.3. Состав и группы созданных признаков (Created Feature Groups)**

In [ ]:
feature_groups = get_feature_groups()

feature_group_summary = pd.DataFrame(
    [
        {
            "feature_group": group,
            "feature_count": len(columns),
        }
        for group, columns in feature_groups.items()
    ]
)
feature_group_summary.to_csv(TABLES_DIR / "feature_group_summary.csv", index=False)
feature_group_details = pd.DataFrame(
    [
        {
            "feature_group": group,
            "feature_name": column,
            "is_present": column in feature_data.columns,
        }
        for group, columns in feature_groups.items()
        for column in columns
    ]
)
feature_group_details.to_csv(TABLES_DIR / "feature_group_details.csv", index=False)

display(Markdown("### **Сводка групп признаков (Feature Group Summary)**"))
display(feature_group_summary)

display(Markdown("### **Детальный состав признаков по группам (Detailed Feature List)**"))
display(feature_group_details)

## **2.4. Проверка временных, календарных и погодных преобразований (Temporal, Calendar, and Weather Checks)**

In [ ]:
inspection_columns = [
    DATETIME_COLUMN,
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday",
    "hour_sin",
    "hour_cos",
    "temp",
    "temp_celsius",
    "weather_main",
    "weather_description",
]
display(Markdown("### **Пример преобразованных признаков (Transformed Feature Sample)**"))
display(feature_data[inspection_columns].head(10))

## **2.5. Проверка причинности lag-признаков (Lag Feature Causality Check)**

In [ ]:
source_target = raw_data.set_index(DATETIME_COLUMN)[TARGET_COLUMN]
audit_row = feature_data.iloc[0]
audit_timestamp = audit_row[DATETIME_COLUMN]

lag_audit = pd.DataFrame(
    {
        "lag_hours": LAG_HOURS,
        "source_timestamp": [audit_timestamp - pd.Timedelta(hours=lag) for lag in LAG_HOURS],
        "source_value": [
            source_target.loc[audit_timestamp - pd.Timedelta(hours=lag)] for lag in LAG_HOURS
        ],
        "generated_value": [audit_row[column] for column in LAG_FEATURE_COLUMNS],
    }
)
lag_audit["source_precedes_target"] = lag_audit["source_timestamp"] < audit_timestamp
lag_audit["values_match"] = lag_audit["source_value"] == lag_audit["generated_value"]

display(Markdown("### **Аудит lag-признаков (Lag Feature Audit)**"))
display(
    Markdown(f"Прогнозируемый интервал: `{audit_timestamp}`; target: `{audit_row[TARGET_COLUMN]}`")
)
display(lag_audit)

## **2.6. Проверка причинности rolling-признаков (Rolling Feature Causality Check)**

In [ ]:
rolling_audit_rows = []
for window in ROLLING_WINDOWS:
    window_end = audit_timestamp - pd.Timedelta(hours=1)
    window_start = audit_timestamp - pd.Timedelta(hours=window)

    past_values = source_target.loc[window_start:window_end]
    expected_mean = past_values.mean()
    generated_mean = audit_row[f"{TARGET_COLUMN}_rolling_mean_{window}"]
    expected_std = past_values.std(ddof=0)
    generated_std = audit_row[f"{TARGET_COLUMN}_rolling_std_{window}"]
    rolling_audit_rows.append(
        {
            "window_hours": window,
            "window_start": window_start,
            "window_end": window_end,
            "latest_value_precedes_target": window_end < audit_timestamp,
            "expected_mean": expected_mean,
            "generated_mean": generated_mean,
            "mean_values_match": np.isclose(expected_mean, generated_mean),
            "expected_std": expected_std,
            "generated_std": generated_std,
            "std_values_match": np.isclose(expected_std, generated_std),
        }
    )

rolling_audit = pd.DataFrame(rolling_audit_rows)
display(Markdown("### **Аудит rolling-признаков (Rolling Feature Audit)**"))
display(rolling_audit)

## **2.7. Контроль финальной очистки набора признаков (Final Feature Cleanup Check)**

In [ ]:
cleanup_summary = pd.DataFrame(
    {
        "check": [
            "missing_target_rows",
            "rows_with_missing_historical_features",
            "chronological_order_preserved",
            "index_reset",
        ],
        "value": [
            int(feature_data[TARGET_COLUMN].isna().sum()),
            int(feature_data[list(HISTORICAL_FEATURE_COLUMNS)].isna().any(axis=1).sum()),
            feature_data[DATETIME_COLUMN].is_monotonic_increasing,
            feature_data.index.equals(pd.RangeIndex(len(feature_data))),
        ],
    }
)

display(Markdown("### **Результаты финальной очистки (Final Cleanup Results)**"))
display(cleanup_summary)

## **2.8. Анализ и интерпретация результатов формирования признаков (Analysis and Interpretation of Feature Engineering Results)**

На этапе формирования признаков подготовленный почасовой набор данных Metro Interstate Traffic Volume был преобразован в расширенную таблицу признаков, пригодную для последующего хронологического разбиения, кодирования, масштабирования и обучения регрессионных моделей. 

**Ключевые результаты:**
1. __Сформированы временные и календарные признаки.__  
   Из поля `date_time` были получены признаки `hour`, `day_of_week`, `month`, а также календарные индикаторы `is_weekend` и `is_holiday`. Эти признаки позволяют модели учитывать суточные, недельные и календарные закономерности транспортной нагрузки.
2. __Реализовано циклическое кодирование периодических компонентов времени.__  
   Для часа суток, дня недели и месяца были сформированы пары признаков `hour_sin` / `hour_cos`, `day_of_week_sin` / `day_of_week_cos`, `month_sin` / `month_cos`. Такое представление устраняет искусственные разрывы между соседними периодами, например между 23:00 и 00:00, воскресеньем и понедельником, декабрем и январем.
3. __Погодные признаки подготовлены без преждевременного кодирования и масштабирования.__  
   Числовые погодные признаки сохранены в исходном виде, дополнительно сформирован признак `temp_celsius` для интерпретируемого представления температуры. Категориальные погодные признаки `weather_main` и `weather_description` сохранены для последующего кодирования после хронологического разбиения выборки. One-Hot Encoding и масштабирование на данном этапе не выполнялись, что позволяет избежать утечки статистики между обучающей, валидационной и тестовой выборками.
4. __Сформированы лаговые признаки транспортной нагрузки.__  
   Для целевой переменной `traffic_volume` были рассчитаны лаги на 1, 2, 3, 24 и 168 часов. Эти признаки отражают краткосрочную, суточную и недельную зависимость текущей нагрузки от предыдущих значений временного ряда.
5. __Сформированы скользящие статистики без утечки целевой переменной.__  
   Скользящие средние и стандартные отклонения рассчитаны только по значениям, предшествующим прогнозируемому интервалу. Использование сдвига перед rolling-расчетом гарантирует, что текущее значение `traffic_volume(t)` не попадает в признаки строки `t`.
6. __Финальная очистка выполнена после построения исторических признаков.__  
   Технические строки с неполными lag- и rolling-признаками были удалены только после формирования полного признакового пространства. Такой порядок сохраняет корректную временную структуру ряда на этапе построения исторических признаков и исключает использование будущих значений.

__Итоговое методологическое резюме:__ Feature Engineering подтвердил практическую реализуемость признакового пространства. Полученная таблица признаков сохраняет причинную структуру временного ряда, не использует текущее или будущее значение целевой переменной при построении исторических признаков и формирует основу для последующего обучения и сравнения моделей прогнозирования транспортной нагрузки.